In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPO = Path.cwd()
if not (REPO / "downstream_tasks").exists():
    REPO = Path("/home/jovyan/dpanc/GENA_LM/GENA_LM_expression_branch")

BENCHMARK_ROOT = Path("/home/jovyan/dpanc/benchmarking")
DATA = BENCHMARK_ROOT / "data"
GENA_ROOT = BENCHMARK_ROOT / "GENA_LM"
ALPHAGENOME_ROOT = BENCHMARK_ROOT / "AlphaGenome"

sys.path.insert(0, str(REPO / "downstream_tasks/expression_prediction/gena_lm_benchmark/scripts"))
from score_ct_specificity import score_predictions


In [2]:
model = "glioma"
split = "test"  # valid or test
cell_set = "json14"

pred_path = GENA_ROOT / "predictions_results" / model / f"gena_lm_{split}_{cell_set}_predictions.csv"
true_path = DATA / f"{split}_true_human.csv"
selected_targets_path = DATA / "selected_targets.csv"

pred = pd.read_csv(pred_path)
true = pd.read_csv(true_path)

if "gene_id" not in pred.columns:
    pred = pred.rename(columns={pred.columns[0]: "gene_id"})
if "gene_id" not in true.columns:
    true = true.rename(columns={true.columns[0]: "gene_id"})

print("pred:", pred.shape, pred_path)
print("true:", true.shape, true_path)
display(pred.head())
display(true.head())


pred: (2781, 15) /home/jovyan/dpanc/benchmarking/GENA_LM/predictions_results/glioma/gena_lm_test_json14_predictions.csv
true: (2781, 15) /home/jovyan/dpanc/benchmarking/data/test_true_human.csv


,gene_id,ENCFF035CWS,ENCFF083EOC,ENCFF123KIW,ENCFF236XOK,ENCFF242BWW,ENCFF329ENM,ENCFF361XCF,ENCFF494KRC,ENCFF602HCV,ENCFF660EXG,ENCFF664WLU,ENCFF761SPP,ENCFF784MDF,ENCFF857JQM
0,ENSG00000232721.2,0.041260,0.082520,0.061035,0.070312,0.052979,0.031494,0.073242,0.070801,0.053223,0.049316,0.062012,0.069824,0.064453,0.074219
1,ENSG00000263590.2,0.006622,0.000610,0.017212,0.032959,0.028809,0.008728,0.025879,0.006195,0.018799,0.002060,0.013000,0.007629,0.018188,0.026001
2,ENSG00000234277.2,0.050049,0.067383,0.038574,0.055664,0.047119,0.058350,0.059814,0.065918,0.063477,0.055176,0.039062,0.069336,0.063965,0.059814
3,ENSG00000181450.17,1.039062,0.832031,0.695312,0.890625,0.714844,0.781250,0.589844,0.621094,0.781250,0.648438,0.695312,0.761719,0.679688,0.820312
4,ENSG00000143740.14,1.398438,1.843750,1.437500,1.593750,1.539062,1.640625,1.804688,2.296875,1.875000,1.718750,1.906250,1.539062,1.851562,1.593750


,gene_id,ENCFF035CWS,ENCFF083EOC,ENCFF123KIW,ENCFF236XOK,ENCFF242BWW,ENCFF329ENM,ENCFF361XCF,ENCFF494KRC,ENCFF602HCV,ENCFF660EXG,ENCFF664WLU,ENCFF761SPP,ENCFF784MDF,ENCFF857JQM
0,ENSG00000001036.13,0.693147,0.463734,3.126760,3.319987,3.291010,1.686399,1.646734,2.520917,3.314186,4.277083,1.710188,4.730392,2.728506,1.178655
1,ENSG00000003137.8,0.190620,0.019803,0.587787,1.800058,0.615186,0.104360,0.157004,0.104360,1.752672,0.095310,0.131028,0.148420,1.007958,0.000000
2,ENSG00000004809.13,0.009950,0.000000,0.223144,0.000000,0.357674,0.000000,0.000000,0.104360,0.019803,0.029559,0.000000,0.000000,0.000000,0.048790
3,ENSG00000005436.13,1.150572,0.746688,2.250239,1.896119,1.798404,0.845868,1.011601,2.316488,2.419479,2.613739,1.406097,3.519573,2.086914,0.993252
4,ENSG00000005448.16,0.593327,0.438255,2.128232,1.867176,2.694627,0.157004,0.989541,2.371178,2.468100,0.667829,0.506818,0.727549,0.476234,0.398776


In [ ]:
def correlation_table(true_df, pred_df):
    true = true_df.set_index("gene_id")
    pred = pred_df.set_index("gene_id")

    common_genes = true.index.intersection(pred.index)
    common_cells = true.columns.intersection(pred.columns)

    true = true.loc[common_genes, common_cells]
    pred = pred.loc[common_genes, common_cells]

    rows = []
    for cell in common_cells:
        true_vec = true[cell].astype(float).values
        pred_vec = pred[cell].astype(float).values
        mask = np.isfinite(true_vec) & np.isfinite(pred_vec)
        true_vec = true_vec[mask]
        pred_vec = pred_vec[mask]
        if len(true_vec) < 2 or np.std(true_vec) == 0 or np.std(pred_vec) == 0:
            corr = np.nan
        else:
            corr = np.corrcoef(true_vec, pred_vec)[0, 1]
        rows.append({"cell_type": cell, "corr_genes": corr})

    result = pd.DataFrame(rows)

    gene_corrs = []
    skipped_genes = []
    for gene in common_genes:
        true_vec = true.loc[gene].astype(float).values
        pred_vec = pred.loc[gene].astype(float).values
        mask = np.isfinite(true_vec) & np.isfinite(pred_vec)
        true_vec = true_vec[mask]
        pred_vec = pred_vec[mask]
        if len(true_vec) < 4 or np.std(true_vec) == 0 or np.std(pred_vec) == 0:
            skipped_genes.append(gene)
            continue
        corr = np.corrcoef(true_vec, pred_vec)[0, 1]
        if np.isfinite(corr):
            gene_corrs.append(corr)
        else:
            skipped_genes.append(gene)

    mean_corr_cells = float(np.mean(gene_corrs)) if gene_corrs else np.nan
    result["corr_cells"] = mean_corr_cells

    mean_row = pd.DataFrame([{
        "corr_genes": result["corr_genes"].mean(),
        "corr_cells": mean_corr_cells,
    }])
    result = pd.concat([result, mean_row], ignore_index=True)

    return result, true.reset_index(), pred.reset_index(), common_genes, common_cells, skipped_genes


In [6]:
result, true_aligned, pred_aligned, common_genes, common_cells, skipped_genes = correlation_table(true, pred)
print("common genes:", len(common_genes))
print("common cells:", len(common_cells))
display(result)


common genes: 2781
common cells: 14


,cell_type,corr_genes,corr_cells
0,ENCFF035CWS,0.704036,0.139641
1,ENCFF083EOC,0.606085,0.139641
2,ENCFF123KIW,0.763124,0.139641
3,ENCFF236XOK,0.766655,0.139641
4,ENCFF242BWW,0.761789,0.139641
5,ENCFF329ENM,0.696262,0.139641
6,ENCFF361XCF,0.736003,0.139641
7,ENCFF494KRC,0.744179,0.139641
8,ENCFF602HCV,0.788167,0.139641
9,ENCFF660EXG,0.770629,0.139641


In [5]:
score_dict = score_predictions(true_aligned, pred_aligned, str(selected_targets_path), need_log=False)
deviation_r = score_dict.get("deviation_r", float("nan"))
print("deviation_r:", deviation_r)


deviation_r: 0.39068257188689537
